# Cygnus: read-only TESS reanalysis pilot

**Research pilot, not a candidate classifier or discovery claim.** Review code before mounting Drive. Enter the actual repo and pack paths and select one recorded product. Nothing below writes into the source pack, runs rclone, downloads archive products, or publishes results. The Colab account may not see products made by a different `drive.file`-scoped connection. See `docs/ANALYSIS_SUITE.md` and `AGENTS.md` for evidence and privacy policy.


In [ ]:
# This cell explicitly grants the runtime access to the signed-in Google Drive.
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path
# EDIT BOTH paths after checking the actual mounted directories. No default is assumed.
REPO_ROOT = Path('SET_REVIEWED_REPO_PATH')
PACK_ROOT = Path('SET_MOUNTED_TIER1_ROOT')
PRODUCT_ID = 'SET_ONE_TESS_SPOC_LC_PRODUCT_ID'
if not (REPO_ROOT / 'pyproject.toml').is_file():
    raise FileNotFoundError('Set REPO_ROOT to a reviewed Cygnus checkout')
if not (PACK_ROOT / '01_mast').is_dir():
    raise FileNotFoundError('Set PACK_ROOT to the verified Tier-1 root on mounted Drive')
if PRODUCT_ID.startswith('SET_'):
    raise ValueError('Choose exactly one TESS SPOC LC product ID from the manifest')


In [ ]:
import subprocess, sys
# Installation executes code from REPO_ROOT; only run after reviewing that checkout.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', str(REPO_ROOT) + '[analysis]'], check=True)
from cygnus.analysis.io import read_manifest, checked_product_path, read_spoc_lightcurve
products = read_manifest(REPO_ROOT / 'docs' / 'tier1_pack' / 'MASTER_MANIFEST.csv')
matches = [r for r in products if r.service == 'mast' and r.product_id == PRODUCT_ID
           and r.dest_rel.startswith('tess_spoc_lc/') and r.dest_rel.endswith('_lc.fits')]
if len(matches) != 1:
    raise ValueError('Product ID must uniquely identify one manifest-listed SPOC LC')
product = matches[0]
source = checked_product_path(PACK_ROOT, product)
print('Manifest product:', product.product_id, 'SHA-256:', product.sha256)


In [ ]:
import hashlib, shutil, tempfile
# Copy to the ephemeral VM; avoid repeated FITS reads over the Drive mount.
scratch = Path(tempfile.mkdtemp(prefix='cygnus-reanalysis-'))
local = scratch / source.name
shutil.copyfile(source, local)
digest = hashlib.sha256()
with local.open('rb') as fh:
    for block in iter(lambda: fh.read(1024 * 1024), b''):
        digest.update(block)
copied_hash = digest.hexdigest()
if copied_hash.lower() != product.sha256.lower():
    local.unlink(missing_ok=True)
    raise ValueError('Copied product failed SHA-256 validation')
lc = read_spoc_lightcurve(local)
print('Identity:', lc['identity'], 'Timing header:', lc['timing'])
# Confirm these header values before interpreting time coordinates or comparing sectors.


In [ ]:
import numpy as np
time = np.asarray(lc['time'], dtype=float)
sap = np.asarray(lc['sap_flux'], dtype=float)
pdc = np.asarray(lc['pdcsap_flux'], dtype=float)
quality = np.asarray(lc['quality'])
good = (quality == 0) & np.isfinite(time) & np.isfinite(sap) & np.isfinite(pdc) & (sap > 0) & (pdc > 0)
if good.sum() < 20:
    raise ValueError('Insufficient quality-zero common cadences for this illustrative comparison')
sap_rel = sap[good] / np.median(sap[good])
pdc_rel = pdc[good] / np.median(pdc[good])
difference = sap_rel - pdc_rel
print({'rows_total': len(time), 'rows_joint_good': int(good.sum()),
       'median_abs_normalized_reduction_difference': float(np.median(np.abs(difference)))})
# This is a reduction disagreement, NOT an astrophysical event or calibrated significance.


In [ ]:
from cygnus.analysis.timeseries import mine_reduction_disagreements
if not lc['timing']['TIMESYS'] or not lc['timing']['TIMEUNIT']:
    raise ValueError('FITS timing standard/unit absent; resolve time provenance before event comparison')
result = mine_reduction_disagreements(time[good], {'SAP_normalized': sap_rel, 'PDCSAP_normalized': pdc_rel},
    flux_unit='relative_flux', time_unit=str(lc['timing']['TIMEUNIT']),
    time_scale=str(lc['timing']['TIMESYS']), max_candidates=5)
print('Triage:', result.status, result.reason, 'usable cadences:', result.usable_cadences)
for item in result.candidates:
    print(item.peak_index, item.peak_time, item.peak_spread, item.rank_score, item.status)
# This ranks SAP/PDC disagreements, not evidence of a transit or novelty.


## Next evidence steps
Use the tested functions in `cygnus.analysis` for a predeclared event window, alternative apertures, image/catalog checks and Gaia NSS *only when the corresponding data and metadata exist*. Freeze controls and a held-out epoch, record all trials, inject synthetic signals through the full pipeline, and audit artifacts and current catalogs. A missing check is `not_tested`. Do not upload unchanged FITS files or generated claims to a public site. Remove the temporary VM copy when finished (`local.unlink()`); Colab VM storage is ephemeral.
